# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"

# Day we're updating data
update_date = "05-29-2025"

# Date range
date_range = "01-01-2023--05-29-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

os.chdir(saved)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Get rid of missing dates; they won't be counted anyway
# for date in metadata["Collection_Date"]:
#     if "/" in date or date == "missing":
#         metadata = metadata[metadata["Collection_Date"] != date]

# # Find only >= 2024 to start
# metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2021, 11, 1).strftime("%Y-%m-%d")] # Note that those with only years will default to today

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2023, 1, 1).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 5, 29).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)

9353
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
18706


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,",",SRS17903639,False,NaN,Massachusetts
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,"Swab, Tracheal",SRS17903639,False,NaN,"USA: Massachusetts, Barnstable County"
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,",/",SRS17903636,False,NaN,Kentucky
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,"Swab Pool, Cloacal/Oropharyngeal",SRS17903636,False,NaN,"USA: Kentucky, Henderson County"
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,2023-06-07 02:01:28,1,22-005158-001,SRP441379,H5N1,NaN,SRS17903631,False,NaN,Maine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18701,SRR33565489,WGS,148.31,58438942,PRJNA1102327,SAMN48491499,Viral,22852781,USDA-NVSL,2025,...,2025-05-14 14:10:53,1,25-013627-005,SRP503016,NaN,NaN,SRS25033260,False,NaN,
18702,SRR33565490,WGS,148.05,73125034,PRJNA1102327,SAMN48491498,Viral,28848992,USDA-NVSL,2025,...,2025-05-14 14:10:50,1,25-013627-004,SRP503016,NaN,milk,SRS25033259,False,NaN,USA
18703,SRR33565490,WGS,148.05,73125034,PRJNA1102327,SAMN48491498,Viral,28848992,USDA-NVSL,2025,...,2025-05-14 14:10:50,1,25-013627-004,SRP503016,NaN,NaN,SRS25033259,False,NaN,
18704,SRR33565491,WGS,146.78,74601613,PRJNA1102327,SAMN48491497,Viral,29227804,USDA-NVSL,2025,...,2025-05-14 14:10:55,1,25-013627-002,SRP503016,NaN,milk,SRS25033258,False,NaN,USA


In [3]:
# Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["C2.1"] # ["B3.13", "D1.1"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

               Run Assay Type  AvgSpotLen     Bases    BioProject  \
0      SRR24839058   AMPLICON      230.77  32942716   PRJNA980729   
1      SRR24839058   AMPLICON      230.77  32942716   PRJNA980729   
2      SRR24839059   AMPLICON      207.54  53993591   PRJNA980729   
3      SRR24839059   AMPLICON      207.54  53993591   PRJNA980729   
4      SRR24839060   AMPLICON      201.66  21703662   PRJNA980729   
...            ...        ...         ...       ...           ...   
18701  SRR33565489        WGS      148.31  58438942  PRJNA1102327   
18702  SRR33565490        WGS      148.05  73125034  PRJNA1102327   
18703  SRR33565490        WGS      148.05  73125034  PRJNA1102327   
18704  SRR33565491        WGS      146.78  74601613  PRJNA1102327   
18705  SRR33565491        WGS      146.78  74601613  PRJNA1102327   

          BioSample BioSampleModel     Bytes  \
0      SAMN35647642          Viral  18373327   
1      SAMN35647642          Viral  18373327   
2      SAMN35647620        

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
4664,SRR29740403,WGS,146.16,117897469,PRJNA980729,SAMN42286250,Viral,41910982,USDA-NVSL,2024-03-11,...,Massachusetts,SRR29740403,2025-05-09_10-48-22,SRR29740403.fa,C2.1,"PB1:am20, HA:ea2, NA:ea2, PA:am3, MP:ea1, NS:e...","am20:23-036657-001:PB1, ea2:23-030074-013:HA, ...","99.69%, 99.47%, 99.57%, 99.67%, 99.29%, 99.05%...","7, 9, 6, 7, 7, 8, 9, 5",Ran on FASTA - No Coverage Report
4665,SRR29740403,WGS,146.16,117897469,PRJNA980729,SAMN42286250,Viral,41910982,USDA-NVSL,2024-03-11,...,"USA: Essex, MA",SRR29740403,2025-05-09_10-48-22,SRR29740403.fa,C2.1,"PB1:am20, HA:ea2, NA:ea2, PA:am3, MP:ea1, NS:e...","am20:23-036657-001:PB1, ea2:23-030074-013:HA, ...","99.69%, 99.47%, 99.57%, 99.67%, 99.29%, 99.05%...","7, 9, 6, 7, 7, 8, 9, 5",Ran on FASTA - No Coverage Report
4666,SRR29740404,WGS,146.14,79862354,PRJNA980729,SAMN42286249,Viral,28646248,USDA-NVSL,2024-03-04,...,Massachusetts,SRR29740404,2025-05-09_10-48-22,SRR29740404.fa,C2.1,"MP:ea1, PB2:am19, HA:ea2, NP:ea1, PB1:am20, NA...","ea1:22-003707-003:MP, am19:23-036193-005:PB2, ...","99.49%, 99.69%, 99.59%, 99.43%, 99.52%, 99.64%...","5, 7, 7, 7, 11, 5, 9, 10",Ran on FASTA - No Coverage Report
4667,SRR29740404,WGS,146.14,79862354,PRJNA980729,SAMN42286249,Viral,28646248,USDA-NVSL,2024-03-04,...,"USA: Brewster, MA",SRR29740404,2025-05-09_10-48-22,SRR29740404.fa,C2.1,"MP:ea1, PB2:am19, HA:ea2, NP:ea1, PB1:am20, NA...","ea1:22-003707-003:MP, am19:23-036193-005:PB2, ...","99.49%, 99.69%, 99.59%, 99.43%, 99.52%, 99.64%...","5, 7, 7, 7, 11, 5, 9, 10",Ran on FASTA - No Coverage Report
4668,SRR29740405,WGS,146.14,30425186,PRJNA980729,SAMN42286247,Viral,10857873,USDA-NVSL,2024-03-01,...,Massachusetts,SRR29740405,2025-05-09_10-48-22,SRR29740405.fa,C2.1,"HA:ea2, NP:ea1, MP:ea1, PA:am3, NA:ea2, PB1:am...","ea2:23-030074-013:HA, ea1:22-003707-003:NP, ea...","99.65%, 99.43%, 99.49%, 99.53%, 99.57%, 99.60%...","6, 7, 5, 10, 6, 9, 10, 6",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4977,SRR29740559,WGS,144.54,24437184,PRJNA980729,SAMN42286253,Viral,8807095,USDA-NVSL,2024-03-11,...,"USA: Nahant, MA",SRR29740559,2025-05-09_10-48-33,SRR29740559.fa,C2.1,"NA:ea2, PA:am3, MP:ea1, HA:ea2, NP:ea1, PB1:am...","ea2:23-004612-024:NA, am3:23-036657-001:PA, ea...","99.50%, 99.67%, 99.29%, 99.59%, 99.35%, 99.69%...","7, 7, 7, 7, 8, 7, 8, 2",Ran on FASTA - No Coverage Report
4978,SRR29740560,WGS,146.15,70500053,PRJNA980729,SAMN42286251,Viral,25051718,USDA-NVSL,2024-03-11,...,Massachusetts,SRR29740560,2025-05-09_10-45-47,SRR29740560.fa,C2.1,"NA:ea2, HA:ea2, PB1:am20, NS:ea1, MP:ea1, PB2:...","ea2:23-004612-024:NA, ea2:23-030074-013:HA, am...","99.50%, 99.59%, 99.69%, 99.05%, 99.29%, 99.91%...","7, 7, 7, 8, 7, 2, 8, 7",Ran on FASTA - No Coverage Report
4979,SRR29740560,WGS,146.15,70500053,PRJNA980729,SAMN42286251,Viral,25051718,USDA-NVSL,2024-03-11,...,"USA: Nahant , MA",SRR29740560,2025-05-09_10-45-47,SRR29740560.fa,C2.1,"NA:ea2, HA:ea2, PB1:am20, NS:ea1, MP:ea1, PB2:...","ea2:23-004612-024:NA, ea2:23-030074-013:HA, am...","99.50%, 99.59%, 99.69%, 99.05%, 99.29%, 99.91%...","7, 7, 7, 8, 7, 2, 8, 7",Ran on FASTA - No Coverage Report
4980,SRR29740561,WGS,146.32,71236974,PRJNA980729,SAMN42286252,Viral,25231921,USDA-NVSL,2024-03-11,...,Massachusetts,SRR29740561,2025-05-09_10-48-00,SRR29740561.fa,C2.1,"NA:ea2, NS:ea1, MP:ea1, NP:ea1, PB1:am20, HA:e...","ea2:23-004612-024:NA, ea1:22-003707-003:NS, ea...","99.57%, 99.05%, 99.29%, 99.35%, 99.69%, 99.53%...","6, 8, 7, 8, 7, 8, 7, 5",Ran on FASTA - No Coverage Report


In [5]:
# Get specific geolocation and name_state from genbank_mapping.tsv

# genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
# genbank_mapping["Run"] = genbank_mapping["sra_run"]
# genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
# genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)


print(metadata["name_state"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

4664        Massachusetts
4665       USA: Essex, MA
4666        Massachusetts
4667    USA: Brewster, MA
4668        Massachusetts
              ...        
4977      USA: Nahant, MA
4978        Massachusetts
4979     USA: Nahant , MA
4980        Massachusetts
4981       USA: Essex, MA
Name: name_state, Length: 254, dtype: object
254


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
4664,SRR29740403,WGS,146.16,117897469,PRJNA980729,SAMN42286250,Viral,41910982,USDA-NVSL,2024-03-11,...,SRR29740403,2025-05-09_10-48-22,SRR29740403.fa,C2.1,"PB1:am20, HA:ea2, NA:ea2, PA:am3, MP:ea1, NS:e...","am20:23-036657-001:PB1, ea2:23-030074-013:HA, ...","99.69%, 99.47%, 99.57%, 99.67%, 99.29%, 99.05%...","7, 9, 6, 7, 7, 8, 9, 5",Ran on FASTA - No Coverage Report,USA-MA
4665,SRR29740403,WGS,146.16,117897469,PRJNA980729,SAMN42286250,Viral,41910982,USDA-NVSL,2024-03-11,...,SRR29740403,2025-05-09_10-48-22,SRR29740403.fa,C2.1,"PB1:am20, HA:ea2, NA:ea2, PA:am3, MP:ea1, NS:e...","am20:23-036657-001:PB1, ea2:23-030074-013:HA, ...","99.69%, 99.47%, 99.57%, 99.67%, 99.29%, 99.05%...","7, 9, 6, 7, 7, 8, 9, 5",Ran on FASTA - No Coverage Report,"USA: Essex, MA"
4666,SRR29740404,WGS,146.14,79862354,PRJNA980729,SAMN42286249,Viral,28646248,USDA-NVSL,2024-03-04,...,SRR29740404,2025-05-09_10-48-22,SRR29740404.fa,C2.1,"MP:ea1, PB2:am19, HA:ea2, NP:ea1, PB1:am20, NA...","ea1:22-003707-003:MP, am19:23-036193-005:PB2, ...","99.49%, 99.69%, 99.59%, 99.43%, 99.52%, 99.64%...","5, 7, 7, 7, 11, 5, 9, 10",Ran on FASTA - No Coverage Report,USA-MA
4667,SRR29740404,WGS,146.14,79862354,PRJNA980729,SAMN42286249,Viral,28646248,USDA-NVSL,2024-03-04,...,SRR29740404,2025-05-09_10-48-22,SRR29740404.fa,C2.1,"MP:ea1, PB2:am19, HA:ea2, NP:ea1, PB1:am20, NA...","ea1:22-003707-003:MP, am19:23-036193-005:PB2, ...","99.49%, 99.69%, 99.59%, 99.43%, 99.52%, 99.64%...","5, 7, 7, 7, 11, 5, 9, 10",Ran on FASTA - No Coverage Report,"USA: Brewster, MA"
4668,SRR29740405,WGS,146.14,30425186,PRJNA980729,SAMN42286247,Viral,10857873,USDA-NVSL,2024-03-01,...,SRR29740405,2025-05-09_10-48-22,SRR29740405.fa,C2.1,"HA:ea2, NP:ea1, MP:ea1, PA:am3, NA:ea2, PB1:am...","ea2:23-030074-013:HA, ea1:22-003707-003:NP, ea...","99.65%, 99.43%, 99.49%, 99.53%, 99.57%, 99.60%...","6, 7, 5, 10, 6, 9, 10, 6",Ran on FASTA - No Coverage Report,USA-MA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4977,SRR29740559,WGS,144.54,24437184,PRJNA980729,SAMN42286253,Viral,8807095,USDA-NVSL,2024-03-11,...,SRR29740559,2025-05-09_10-48-33,SRR29740559.fa,C2.1,"NA:ea2, PA:am3, MP:ea1, HA:ea2, NP:ea1, PB1:am...","ea2:23-004612-024:NA, am3:23-036657-001:PA, ea...","99.50%, 99.67%, 99.29%, 99.59%, 99.35%, 99.69%...","7, 7, 7, 7, 8, 7, 8, 2",Ran on FASTA - No Coverage Report,"USA: Nahant, MA"
4978,SRR29740560,WGS,146.15,70500053,PRJNA980729,SAMN42286251,Viral,25051718,USDA-NVSL,2024-03-11,...,SRR29740560,2025-05-09_10-45-47,SRR29740560.fa,C2.1,"NA:ea2, HA:ea2, PB1:am20, NS:ea1, MP:ea1, PB2:...","ea2:23-004612-024:NA, ea2:23-030074-013:HA, am...","99.50%, 99.59%, 99.69%, 99.05%, 99.29%, 99.91%...","7, 7, 7, 8, 7, 2, 8, 7",Ran on FASTA - No Coverage Report,USA-MA
4979,SRR29740560,WGS,146.15,70500053,PRJNA980729,SAMN42286251,Viral,25051718,USDA-NVSL,2024-03-11,...,SRR29740560,2025-05-09_10-45-47,SRR29740560.fa,C2.1,"NA:ea2, HA:ea2, PB1:am20, NS:ea1, MP:ea1, PB2:...","ea2:23-004612-024:NA, ea2:23-030074-013:HA, am...","99.50%, 99.59%, 99.69%, 99.05%, 99.29%, 99.91%...","7, 7, 7, 8, 7, 2, 8, 7",Ran on FASTA - No Coverage Report,"USA: Nahant , MA"
4980,SRR29740561,WGS,146.32,71236974,PRJNA980729,SAMN42286252,Viral,25231921,USDA-NVSL,2024-03-11,...,SRR29740561,2025-05-09_10-48-00,SRR29740561.fa,C2.1,"NA:ea2, NS:ea1, MP:ea1, NP:ea1, PB1:am20, HA:e...","ea2:23-004612-024:NA, ea1:22-003707-003:NS, ea...","99.57%, 99.05%, 99.29%, 99.35%, 99.69%, 99.53%...","6, 8, 7, 8, 7, 8, 7, 5",Ran on FASTA - No Coverage Report,USA-MA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Get and save collection date

In [7]:

# # Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

In [8]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv")
# os.chdir(temp_files)

# # Get only updated dates

# unknown_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] == "2024") | (metadata_genbank["Collection_Date_Specific"] == "2025")] # Dates we don't have
# known_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] != "2024") & (metadata_genbank["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# # Get new dates also 
# # new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata_genbank[["Collection_Date_Specific"]])

# display(metadata_genbank)

In [9]:
# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [10]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['red-tailed hawk', 'bald eagle', 'corvus ossifragus', 'lesser scaup', '-', 'common raven', 'sanderling', 'white-winged scoter', 'common loon', 'great black-backed gulll', 'american crow', 'great black-backed gull', 'scoter', 'great horned owl', 'common eider', nan, 'herring gull', 'black scoter', 'brandt goose', 'canada goose', 'surf scoter']
['-', nan]
                        avian               cattle        feline  \
0            great_horned_owl            dairy_cow           cat   
1                common_raven               cattle  domestic_cat   
2               cooper's_hawk  cattle milk product     feral_cat   
3                coopers_hawk          bovine_milk        feline   
4                     peafowl              bovine   domestic-cat   
..                        ...                  ...           ...   
404  great black-backed gulll                  NaN           NaN   
405               common loon                  NaN           NaN   
406       white-winged scoter  

In [11]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [12]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    
    if collection_date != collection_date: # If nan
        metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata.loc[num, "Collection_Date"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata.loc[num, "Collection_Date"] = date

    metadata = metadata.dropna(thresh=2)

# Make names

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["isolate"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata[["ReleaseDate", 'create_date', 'Collection_Date']])

,ReleaseDate,create_date,Collection_Date
4664,2024-07-08,2024-07-08 16:11:26,2024-03-11
4665,2024-07-08,2024-07-08 16:11:26,2024-03-11
4666,2024-07-08,2024-07-08 16:11:34,2024-03-04
4667,2024-07-08,2024-07-08 16:11:34,2024-03-04
4668,2024-07-08,2024-07-08 16:11:35,2024-03-01
...,...,...,...
4977,2024-07-08,2024-07-08 16:11:58,2024-03-11
4978,2024-07-08,2024-07-08 16:12:08,2024-03-11
4979,2024-07-08,2024-07-08 16:12:08,2024-03-11
4980,2024-07-08,2024-07-08 16:12:16,2024-03-11


In [13]:
print(metadata)

              Run Assay Type  AvgSpotLen        Bases   BioProject  \
4664  SRR29740403        WGS      146.16  117897469.0  PRJNA980729   
4665  SRR29740403        WGS      146.16  117897469.0  PRJNA980729   
4666  SRR29740404        WGS      146.14   79862354.0  PRJNA980729   
4667  SRR29740404        WGS      146.14   79862354.0  PRJNA980729   
4668  SRR29740405        WGS      146.14   30425186.0  PRJNA980729   
...           ...        ...         ...          ...          ...   
4977  SRR29740559        WGS      144.54   24437184.0  PRJNA980729   
4978  SRR29740560        WGS      146.15   70500053.0  PRJNA980729   
4979  SRR29740560        WGS      146.15   70500053.0  PRJNA980729   
4980  SRR29740561        WGS      146.32   71236974.0  PRJNA980729   
4981  SRR29740561        WGS      146.32   71236974.0  PRJNA980729   

         BioSample BioSampleModel       Bytes Center Name Collection_Date  \
4664  SAMN42286250          Viral  41910982.0   USDA-NVSL      2024-03-11   
4665 

## Make FASTA files

In [14]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [15]:
# print(fasta_files.keys())

In [16]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = complete_files + pair + "_andersen_updated_" + update_date + ".fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR29740403|A/-/Massachusetts/24MM00169/2024|H5N1|USA-MA|2024-03-11|other|C2.1
>SRR29740403|A/-/Massachusetts/24MM00169/2024|H5N1|USA-MA|2024-03-11|other|C2.1
nan
nan
nan
nan
nan
nan
>SRR29740407|A/-/Massachusetts/23WI00358/2024|H5N1|USA-MA|2024-02-28|other|C2.1
>SRR29740407|A/-/Massachusetts/23WI00358/2024|H5N1|USA-MA|2024-02-28|other|C2.1
nan
nan
>SRR29740409|A/-/Massachusetts/24HP00063/2024|H5N1|USA-MA|2024-02-23|other|C2.1
>SRR29740409|A/-/Massachusetts/24HP00063/2024|H5N1|USA-MA|2024-02-23|other|C2.1
nan
nan
nan
nan
>SRR29740412|A/-/Massachusetts/24MM00125/2024|H5N1|USA-MA|2024-02-26|other|C2.1
>SRR29740412|A/-/Massachusetts/24MM00125/2024|H5N1|USA-MA|2024-02-26|other|C2.1
>SRR29740414|A/-/Massachusetts/24MM00123/2024|H5N1|USA-MA|2024-02-27|other|C2.1
>SRR29740414|A/-/Massachusetts/24MM00123/2024|H5N1|USA-MA|2024-02-27|other|C2.1
>SRR29740416|A/-/Massachusetts/23WI00356/2024|H5N1|USA-MA|2024-02-27|other|C2.1
>SRR29740416|A/-/Massachusetts/23WI00356/2024|H5N1|USA-MA|2024-02-27|oth

## De-Duplication

In [17]:
# De-duplication 

# Gisaid 

# gisaid = downloads + "GISAID/complete/B3_13_D1_1/" + date_range + "_B3_13_D1_1_North_America/"

gisaid = downloads + "GISAID/complete/2023-01-01--2025-05-29_C2_1_all_continents/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

C2.1_HA
C2.1_MP
C2.1_NA
C2.1_NP
C2.1_NS
C2.1_PA
C2.1_PB1
C2.1_PB2


In [18]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(complete_files)

C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/Andersen/complete/
C:/Users/maksi/Documents/Statistics/Proj

In [19]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [20]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

C2.1_HA
C2.1_MP
C2.1_NA
C2.1_NP
C2.1_NS
C2.1_PA
C2.1_PB1
C2.1_PB2
defaultdict(<class 'list'>, {'C2.1_HA': [    isolate_partial                                        full_header  \
0         24MM00169  >SRR29740403|A/-/Massachusetts/24MM00169/2024|...   
1         24MM00169  >SRR29740403|A/-/Massachusetts/24MM00169/2024|...   
2               NaN                                              nan\n   
3               NaN                                              nan\n   
4               NaN                                              nan\n   
..              ...                                                ...   
249             NaN                                              nan\n   
250             NaN                                              nan\n   
251             NaN                                              nan\n   
252       24MM00170  >SRR29740561|A/-/Massachusetts/24MM00170/2024|...   
253       24MM00170  >SRR29740561|A/-/Massachusetts/24MM00170/2024|...   

    

In [21]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

8
8


In [22]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

254
297
    isolate_partial                                        full_header  \
0        FAV-0237-2  >EPI_ISL_19776301|A/American_Crow/QC/FAV-0237-...   
1        FAV-0237-1  >EPI_ISL_19776300|A/American_Crow/QC/FAV-0237-...   
2        014126-058  >EPI_ISL_19736879|A/sanderling/Massachusetts/0...   
3        014126-066  >EPI_ISL_19736873|A/american_crow/Massachusett...   
4        FAV-0121-1  >EPI_ISL_19776296|A/Peregrine_Falcon/QC/FAV-01...   
..              ...                                                ...   
232       24MM00212  >SRR29740549|A/-/Massachusetts/24MM00212/2024|...   
236       23NE01782  >SRR29740552|A/-/Massachusetts/23NE01782/2024|...   
244             NaN                                              nan\n   
246             NaN                                              nan\n   
252       24MM00170  >SRR29740561|A/-/Massachusetts/24MM00170/2024|...   

                                              sequence  
0    atggagaacatagtactacttcttgcaatagttagccttgt

In [23]:
# If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             # full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)

## Create FASTA files combining Andersen and GISAID

In [24]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

C2.1_HA
C2.1_MP
C2.1_NA
C2.1_NP
C2.1_NS
C2.1_PA
C2.1_PB1
C2.1_PB2
